In [46]:
import pandas as pd
import os
from src.config import *


In [47]:
os.getcwd()
#pd.set_option('display.max_rows', None)  # 행 제한 없음

'c:\\Users\\AICT\\Desktop\\PythonProject\\KNPA_TAD\\src'

In [3]:
df = pd.read_csv('../data/siheung_sim/raw/Siheung_14days_lanes_w_arith.csv')


In [30]:
melted_list = []

# 2. 1~5차로에 대해 순회하며 처리
for i in range(1, 6):
    # 차로 존재 여부 판단을 위한 5개 컬럼 (모두 NaN이면 제외)
    check_cols = [
        f'VEHS(ALL)_{i}', 
        f'SPEEDAVGARITH(ALL)_{i}', 
        f'SPEEDAVGHARM(ALL)_{i}', 
        f'QUEUEDELAY(ALL)_{i}', 
        f'OCCUPRATE(ALL)_{i}'
    ]
    
    # 데이터프레임에 해당 컬럼들이 있는지 확인
    existing_check_cols = [c for c in check_cols if c in df.columns]
    
    if existing_check_cols:
        # 원본에서 시간, 링크 정보와 해당 차로 컬럼들만 복사
        temp_df = df[['TOT_DT', 'LINK_ID'] + existing_check_cols].copy()
        
        # 조건 적용: 5개 지표가 모두 NaN인 행 식별
        mask_all_nan = temp_df[existing_check_cols].isnull().all(axis=1)
        
        # 모두 NaN인 행을 제외한 데이터만 남김
        temp_df = temp_df[~mask_all_nan]
        
        # 결과용 데이터프레임 구성 (컬럼명 변경)
        res_df = pd.DataFrame()
        res_df['TOT_DT'] = temp_df['TOT_DT']
        res_df['LINK_ID'] = temp_df['LINK_ID']
        res_df['LANE_NO'] = i
        res_df['TRF_QNTY'] = temp_df[f'VEHS(ALL)_{i}']
        res_df['AVG_SPD'] = temp_df[f'SPEEDAVGARITH(ALL)_{i}']
        res_df['OCPN_RATE'] = temp_df[f'OCCUPRATE(ALL)_{i}']
        
        melted_list.append(res_df)

# 3. 모든 차로 데이터를 하나로 합침
final_df = pd.concat(melted_list, ignore_index=True)

# 4. 정렬 및 인덱스 초기화
final_df = final_df.sort_values(by=['TOT_DT', 'LINK_ID', 'LANE_NO']).reset_index(drop=True)

final_df = final_df.fillna(0)

In [31]:
print(final_df.isnull().sum())

TOT_DT       0
LINK_ID      0
LANE_NO      0
TRF_QNTY     0
AVG_SPD      0
OCPN_RATE    0
dtype: int64


In [32]:
null_rows = final_df[final_df.isnull().any(axis=1)]
null_rows

,TOT_DT,LINK_ID,LANE_NO,TRF_QNTY,AVG_SPD,OCPN_RATE


In [33]:
final_df.to_csv('../data/siheung_sim/raw/SIHEUNG_SIM_RAW.csv', index=False)

In [54]:
len(final_df)
final_df.head(10)

,TOT_DT,LINK_ID,LANE_NO,TRF_QNTY,AVG_SPD,OCPN_RATE
0,2024-11-29 0:00,2240007900,1,4.0,96.2,0.270
1,2024-11-29 0:00,2240007900,2,12.0,85.8,0.828
2,2024-11-29 0:00,2240007900,3,13.0,90.9,0.988
3,2024-11-29 0:00,2240007900,4,8.0,94.9,0.548
4,2024-11-29 0:00,2240007900,5,1.0,98.3,0.072
5,2024-11-29 0:00,2240009900,1,4.0,103.4,0.216
6,2024-11-29 0:00,2240009900,2,15.0,92.4,1.056
7,2024-11-29 0:00,2240009900,3,11.0,83.6,0.822
8,2024-11-29 0:00,2240009900,4,4.0,97.0,0.310
9,2024-11-29 0:00,2240010000,1,9.0,102.5,0.258


In [ ]:
import pandas as pd

def split_dataset_siheung_sim(data_path, infer=False, seq_len=30, tr_ratio=0.7, val_ratio=0.2, te_ratio=0.1, event_rules=None, start_time=None):
    """
    시흥시 시뮬레이션 교통 데이터셋 분할(Tr, Val, Te, Infer)  함수
    """
    
    # 데이터 경로 설정
    raw_path    = DATA_PATH[data_path]['raw']
    tr_path     = DATA_PATH[data_path]['tr']
    val_path    = DATA_PATH[data_path]['val']
    te_path     = DATA_PATH[data_path]['te']
    infer_path  = DATA_PATH[data_path]['infer']

    print(f"Raw 데이터 로드: {raw_path}")
    df = pd.read_csv(raw_path) 
    
    # Text 타입의 TOT_DT datetime 타입으로의 변환
    df['TOT_DT'] = pd.to_datetime(df['TOT_DT'], errors='coerce')
    
    # 데이터셋 분할 및 저장
    group_by_cols = [col for col in GRP_COLS if col != 'TOT_DT']

    if infer:
        print("추론용 데이터 추출 모드...")
        infer_df_list = []
        groups = df.groupby(group_by_cols) 
        
        for _, group in groups:
            group_sorted = group.sort_values(by='TOT_DT') 
            latest_data = group_sorted.tail(seq_len)
            infer_df_list.append(latest_data)
            
        infer_df = pd.concat(infer_df_list)
        infer_df.to_csv(infer_path, index=False)
        print(f"추론용 데이터 저장 완료: {infer_path}")

    else:
        print("데이터 분할 모드...")
        # 시계열 순서대로 분할하기 위해 정렬 확인
        df = df.sort_values(by='TOT_DT')
        
        total_rows = len(df)
        tr_end = int(total_rows * tr_ratio)
        val_end = tr_end + int(total_rows * val_ratio)
        
        tr_df = df.iloc[:tr_end]
        val_df = df.iloc[tr_end:val_end]
        te_df = df.iloc[val_end:]
        
        tr_df.to_csv(tr_path, index=False)
        val_df.to_csv(val_path, index=False)
        te_df.to_csv(te_path, index=False)
        
        print(f"학습 데이터 저장 완료: {tr_path}")
        print(f"검증 데이터 저장 완료: {val_path}")
        print(f"테스트 데이터 저장 완료: {te_path}")

In [73]:
import pandas as pd

def generate_dataset_siheung_sim(data_path):
    """
    시흥시 시뮬레이션 교통 데이터 전처리 함수(TimeInterval 인코딩, 컬럼 정렬)
    """

    df = pd.read_csv(data_path)

    # Text 타입의 TOT_DT datetime 타입으로의 변환
    df['TOT_DT'] = pd.to_datetime(df['TOT_DT'], errors='coerce')
    
    # 'pred' 열 생성
    df['pred'] = 0 
    
    # TimeInterval 함수 정의
    def _get_time_interval(hour):
        if    0 <= hour <= 7:  return 0
        elif  8 <= hour <= 9:  return 1
        elif 10 <= hour <= 17: return 2
        elif 18 <= hour <= 19: return 3
        elif 20 <= hour <= 23: return 4
        return -1
    
    # TimeInterval 열 생성 (.dt 접근자 사용 가능)
    df['TimeInterval'] = df['TOT_DT'].dt.hour.apply(_get_time_interval)
    
    all_intervals = [0, 1, 2, 3, 4]
    
    # 범주형 변환 및 더미 변수 생성
    df['TimeInterval'] = pd.Categorical(df['TimeInterval'], categories=all_intervals)
    df = pd.get_dummies(df, columns=['TimeInterval'], prefix='TimeInt')

    # 최종 컬럼 정리
    final_timeint_cols = [col for col in df.columns if 'TimeInt' in col]
    final_cols = GRP_COLS + INPUT_COLS + final_timeint_cols + ['pred']
    
    # 컬럼 순서 재배열 및 중복 제거
    df_fin = df.reindex(columns=final_cols, fill_value=0).drop_duplicates()
    
    return df_fin


In [65]:
DATA_PATH = { # 데이터셋 경로
    # raw(원시): data scaling 이전 
    # tr(학습), val(검증), te(테스트), infer(추론): data scaling 이후 TODO: 논의 필요
    'SIHEUNG_SIM': { # 시흥 시뮬레이션 데이터
        'raw':   '../data/siheung_sim/raw/SIHEUNG_SIM_RAW.csv',   
        'tr':    '../data/siheung_sim/train/SIHEUNG_SIM_TR.csv', 
        'val':   '../data/siheung_sim/valid/SIHEUNG_SIM_VAL.csv',
        'te':    '../data/siheung_sim/test/SIHEUNG_SIM_TE.csv',
        'infer': '../data/siheung_sim/inference/SIHEUNG_SIM_INFER.csv',
    }
}

In [74]:
split_dataset_siheung_sim(data_path   = TAD_VER, 
                          infer       = False, 
                          seq_len     = SEQ_LEN, 
                          tr_ratio    = 0.7, 
                          val_ratio   = 0.2, 
                          te_ratio    = 0.1, # 실제 운영시에는 testset 필요없음
                          event_rules = None, 
                          start_time  = None)

Raw 데이터 로드: ../data/siheung_sim/raw/SIHEUNG_SIM_RAW.csv
데이터 분할 모드...
학습 데이터 저장 완료: ../data/siheung_sim/train/SIHEUNG_SIM_TR.csv
검증 데이터 저장 완료: ../data/siheung_sim/valid/SIHEUNG_SIM_VAL.csv
테스트 데이터 저장 완료: ../data/siheung_sim/test/SIHEUNG_SIM_TE.csv


In [75]:
df_train = generate_dataset_siheung_sim(data_path=DATA_PATH[TAD_VER]['tr'])

In [ ]:
df_train